# 07 — Fairness and Responsible-Use Analysis

**Phase 9**, step 1 of 3: fairness and responsible-use analysis. Step 2 will consolidate limitations across the whole project; step 3 (the model card, `reports/model_card.md`) was done first, at explicit request, and is updated after this step with what's found here.

**What this notebook can and can't do.** This dataset has no demographic fields — no race, gender, income, or marital status. So this is not, and cannot be, a formal disparate-impact audit in the regulatory sense (which requires exactly those fields to check outcomes across protected groups). What it *can* do: (1) extend the segment-calibration checks already done for `Region`/`BonusMalus` (Phase 7 step 4) to `DrivAge`, an explicit age variable several jurisdictions restrict; (2) check whether the geographic variables already flagged for miscalibration correlate with age, which would compound one open problem with another; (3) state plainly what cannot be checked here, rather than implying an audit happened when it didn't.

In [1]:
import numpy as np
import pandas as pd

from auto_pricing.frequency import predict_frequency
from auto_pricing.severity import build_severity_table, fit_lognormal_model, predict_lognormal_severity
from auto_pricing.pure_premium import fit_paid_frequency_glm, predict_annual_pure_premium

pp_table = pd.read_parquet("../data/processed/pure_premium_table.parquet")
train = pp_table[pp_table["split"] == "train"].copy()
val = pp_table[pp_table["split"] == "validation"].copy()
print(f"train: {len(train):,}  validation: {len(val):,}")

train: 474,609  validation: 101,702


## 1. Refit the champion pipeline (train), predict on validation

Same champion as Phase 7: paid-frequency GLM x lognormal severity GLM, both refit on train only. Validation, not test, is used here — consistent with Phase 7 step 4's own practice of checking segment-level calibration on validation and reserving test for the one-time overall confirmation already done.

In [2]:
paid_freq_result = fit_paid_frequency_glm(train)
paid_freq_pred_val = predict_frequency(paid_freq_result, val)

sev = pd.read_parquet("../data/processed/severity_clean.parquet")
severity_table = build_severity_table(sev, pp_table.drop(columns=["ClaimNbFromSev", "ClaimAmountSum"]))
sev_train = severity_table[severity_table["split"] == "train"].copy()
severity_result, smearing_factor = fit_lognormal_model(sev_train)
severity_pred_val = predict_lognormal_severity(severity_result, smearing_factor, val)

val_check = val.copy()
val_check["pred"] = paid_freq_pred_val * severity_pred_val
print(f"validation O/E (sanity check against Phase 7 step 4's 0.8430): "
      f"{val_check['ClaimAmountSum'].sum() / val_check['pred'].sum():.4f}")

validation O/E (sanity check against Phase 7 step 4's 0.8430): 0.8430


## 2. DrivAge segment calibration — extending Region/BonusMalus to age

Phase 7 step 4 checked segment-level O/E by `Region` and `BonusMalus`, and found real, unexplained heterogeneity for `Region` (0.46 to 1.83) not attributable to sample size alone. `DrivAge` was never checked this way — it's checked now because it's an explicit age variable, and a model that's badly miscalibrated for a specific age group is a direct fairness concern, not just an accuracy one.

In [3]:
drivage_seg = val_check.groupby("DrivAgeBand", observed=True).agg(
    n_policies=("IDpol", "size"), exposure=("Exposure", "sum"),
    observed=("ClaimAmountSum", "sum"), predicted=("pred", "sum"),
)
drivage_seg["oe_ratio"] = drivage_seg["observed"] / drivage_seg["predicted"]
drivage_seg.round(2)

,n_policies,exposure,observed,predicted,oe_ratio
DrivAgeBand,,,,,
18-22,2481,968.62,490353.01,503354.79,0.97
23-29,10891,4731.51,930583.50,1017174.08,0.91
30-39,25317,12315.04,1652124.43,1896641.29,0.87
40-49,24785,13105.47,1620872.85,2339132.74,0.69
50-59,21475,12019.23,1753294.59,2015186.82,0.87
60-69,10175,6029.93,515705.18,901301.11,0.57
70+,6578,4468.96,947327.04,711064.75,1.33


### 3. Does Region correlate with DrivAge? A compounding check

Phase 7's regional miscalibration is unexplained. If certain regions also skew toward a particular age band, a regional pricing error would land disproportionately on one age group — compounding one open problem with another rather than being two independent issues. Checked directly rather than assumed either way.

In [4]:
region_age = train.groupby("RegionGrouped", observed=True)["DrivAge"].agg(["mean", "median", "count"])
region_age.sort_values("mean").round(1)

,mean,median,count
RegionGrouped,,,
R31,40.9,39.0,19173
R22,41.9,40.0,5631
R23,43.3,41.0,6173
R11,44.6,43.0,48760
R52,44.7,43.0,27136
R82,44.8,44.0,59456
Other,45.1,44.0,14746
R26,45.1,45.0,7310
R73,45.4,44.0,11891


## 4. What cannot be checked here, and why that matters

This dataset carries no race, gender, income, marital-status, or occupation field. That means:

- No correlation between any rating factor and a protected characteristic can be computed — not because the check was skipped, but because the underlying data to run it does not exist in this dataset.
- Any claim that `Region`, `Density`, or `Area` are (or are not) proxies for a protected characteristic is **not something this project can verify**. It can only be flagged as an open question — a real limitation, not a finding.
- A genuine fairness audit of a production version of this model would require either (a) access to protected-class data under appropriate legal safeguards, or (b) a formal proxy methodology (e.g., Bayesian Improved Surname Geocoding or a similar indirect estimation technique) that this project does not attempt, since it would require external reference data this project does not have.

## 5. Jurisdiction and deployment considerations

Stated as background, not as a jurisdiction-specific legal conclusion (which is outside this project's scope and was never checked against any specific jurisdiction's actual rules):

- **Age-based rating** for adult drivers is restricted or banned in some jurisdictions (the EU Gender Directive is the well-known analogue for sex-based rating, though that variable isn't in this dataset at all). Any real deployment of `DrivAge` as a rating factor requires jurisdiction-specific legal review this project did not perform.
- **Territory-based rating** (`Region`/`Area`/`Density` here) faces its own restrictions in some jurisdictions, partly *because* geography can proxy for protected characteristics — exactly the concern section 4 says this project cannot rule out or confirm.
- **`BonusMalus`** is France's own regulated encoding of prior claims history, not a demographic proxy by construction — the most defensible of the strong rating factors used here on fairness grounds, though this framing itself was not independently audited.
- **Deploying without ongoing monitoring** would be a mistake regardless of the above: this model has a confirmed, unresolved regional miscalibration (Phase 7 step 4) that a real deployment would need to monitor for, not just check once at launch.

## Summary of step 1

- `DrivAge` segment calibration was checked for the first time (previously only Region and BonusMalus were) — results and interpretation in `reports/fairness_analysis.md`.
- Region and DrivAge's relationship was checked directly rather than left as an assumption.
- What this project explicitly cannot check (any real proxy-variable analysis, absent demographic data) is disclosed plainly, not glossed over.
- Full write-up: `reports/fairness_analysis.md`. `reports/model_card.md`'s fairness section will be updated to reflect this rather than remain marked preliminary.

### Next
Step 2: consolidate every limitation documented across Phases 2-9 into one reference document, `reports/limitations.md`.